# S11 질의 조건 추출 정확도

목적: `rag.conditions`가 뽑은 조건이 정답 문서 메타데이터와 얼마나 맞는지 잰다(`docs/slices/11-query-conditions.md`). 추출기에는 정답을 넣지 않았고, 여기서만 정답 문서를 기대값으로 쓴다. 검색 점수는 보지 않는다. G3는 이 측정을 한 번 하고, 그 뒤에는 결함만 고친다(사용자 결정). 질의 원문은 오추출 유형마다 몇 건만 보인다(D-10).

이 노트북은 결함 수정(D-21) 전 조건 파일로 실행했다. 첫 표와 오추출 절은 첫 측정 값이고, 수정 효과는 "결함 후보와 수정 효과" 절의 후보별 표다.

In [1]:
import json
from pathlib import Path

import pandas as pd

from rag.conditions import doc_values

In [2]:
ROOT = Path.cwd().parent
CORPUS = ROOT / "data/processed/corpus_context.jsonl"
COND = {"multidoc": ROOT / "data/conditions/multidoc.jsonl",
        "summary_q": ROOT / "data/conditions/summary_q.jsonl"}
FROZEN = ROOT / "data/multidoc/queries_g3.jsonl"
QUERIES = ROOT / "data/processed/queries_summary_q.jsonl"
DEV = ROOT / "data/processed/queries_summary_q.dev.jsonl"
FIELDS = ["date", "generation_number", "meeting_number", "session_number",
          "committee_name", "meeting_name", "names"]

## 데이터 로딩

In [18]:
with CORPUS.open(encoding="utf-8") as f:
    values = {d["doc_id"]: doc_values(d) for d in map(json.loads, f)}
cond = {k: {r["qid"]: r["conditions"] for r in map(json.loads, p.open(encoding="utf-8"))}
        for k, p in COND.items()}
frozen = {q["qid"]: q for q in map(json.loads, FROZEN.open(encoding="utf-8"))}
dev_qids = {json.loads(line)["qid"] for line in DEV.open(encoding="utf-8")}
single = {q["qid"]: q for q in map(json.loads, QUERIES.open(encoding="utf-8"))
          if q["qid"] in dev_qids}
print(len(values), {k: len(v) for k, v in cond.items()}, len(frozen), len(single))

38516 {'multidoc': 600, 'summary_q': 3016} 600 3016


In [19]:
# 질의마다 정답 문서 목록과 유형
gold = {"multidoc": {q: (r["gold_doc_ids"], r["type"]) for q, r in frozen.items()},
        "summary_q": {q: ([r["doc_id"]], "single") for q, r in single.items()}}
{k: len(v) for k, v in gold.items()}

{'multidoc': 600, 'summary_q': 3016}

## 정확도 표

- 발화율: 그 필드가 하나 이상 뽑힌 질의 비율
- 정밀도: 뽑힌 값 중 정답 문서 하나 이상의 값과 맞는 비율
- 적중률: 맞는 값이 하나 이상 뽑힌 질의 비율

In [20]:
rows = []
for name in ["multidoc", "summary_q"]:
    for qid, (docs, qtype) in gold[name].items():
        c = cond[name][qid]
        for field in FIELDS:
            got = c.get(field, [])
            ok = [v for v in got if any(v in values[d][field] for d in docs)]
            rows.append({"set": name, "type": qtype, "field": field, "qid": qid,
                         "fired": bool(got), "n": len(got), "n_ok": len(ok), "hit": bool(ok)})
per = pd.DataFrame(rows)
per.shape

(25312, 8)

In [21]:
def table(df):
    g = df.groupby(["set", "type", "field"])
    t = pd.DataFrame({"질의": g.size() // 1, "발화율": g["fired"].mean(),
                      "정밀도": g["n_ok"].sum() / g["n"].sum(), "적중률": g["hit"].mean()})
    return t.round(3)


table(per)

질의    발화율    정밀도    적중률
set       type       field                                       
multidoc  conf       committee_name      233  0.970  0.996  0.970
                     date                233  1.000  0.930  1.000
                     generation_number   233  0.000    NaN  0.000
                     meeting_name        233  1.000  0.947  1.000
                     meeting_number      233  0.275  0.877  0.245
                     names               233  0.412  0.481  0.245
                     session_number      233  0.262  1.000  0.262
          law        committee_name      152  0.974  1.000  0.974
                     date                152  1.000  1.000  1.000
                     generation_number   152  0.000    NaN  0.000
                     meeting_name        152  0.993  0.956  0.993
                     meeting_number      152  0.539  1.000  0.539
                     names               152  0.283  0.277  0.079
                     session_number      152  0.605  1.000  0.605
          questioner committee_name      215  0.912  0.974  0.888
                     date                215  1.000  0.909  1.000
                     generation_number   215  0.000    NaN  0.000
                     meeting_name        215  0.981  0.968  0.981
                     meeting_number      215  0.456  0.919  0.447
                     names               215  1.000  0.522  1.000
                     session_number      215  0.447  1.000  0.447
summary_q single     committee_name     3016  0.003  0.800  0.003
                     date               3016  0.010  0.158  0.002
                     generation_number  3016  0.000  1.000  0.000
                     meeting_name       3016  0.003  0.600  0.002
                     meeting_number     3016  0.000    NaN  0.000
                     names              3016  0.113  0.054  0.007
                     session_number     3016  0.000  0.000  0.000

## 오추출 유형

정밀도를 깎은 값(뽑혔는데 정답 문서 어디와도 맞지 않는 값)을 필드별로 센다.

In [22]:
wrong = []
for name in ["multidoc", "summary_q"]:
    for qid, (docs, qtype) in gold[name].items():
        for field, got in cond[name][qid].items():
            for v in got:
                if not any(v in values[d][field] for d in docs):
                    wrong.append({"set": name, "type": qtype, "field": field,
                                  "value": v, "qid": qid})
wrong = pd.DataFrame(wrong)
wrong.groupby(["set", "field"]).size()

set        field         
multidoc   committee_name      6
           date               44
           meeting_name       27
           meeting_number     17
           names             348
summary_q  committee_name      2
           date               32
           meeting_name        4
           names             351
           session_number      1
dtype: int64

In [23]:
# 틀린 값 상위(같은 값이 여러 질의에서 틀렸는지)
for (s, field), g in wrong.groupby(["set", "field"]):
    print(s, field, g["value"].value_counts().head(8).to_dict())

multidoc committee_name {'인사청문특별위원회': 5, '교육위원회': 1}
multidoc date {'2019': 11, '2018': 10, '2017': 6, '2021': 5, '2023': 3, '2022': 2, '2020': 2, '2020-05': 1}
multidoc meeting_name {'예산결산특별위원회': 14, '특별위원회': 13}
multidoc meeting_number {'제1회': 10, '제2회': 5, '제3회': 1, '제4회': 1}
multidoc names {'위원': 247, '정부': 49, '소위원장': 8, '이용': 4, '통일부장관': 3, '이원욱': 1, '이수혁': 1, '이재용': 1}
summary_q committee_name {'국방위원회': 1, '교육위원회': 1}
summary_q date {'2020': 4, '2015': 4, '2009': 2, '2013': 2, '2016': 2, '2019': 2, '2002-01': 1, '2003-08': 1}
summary_q meeting_name {'국정감사': 3, '소위원회': 1}
summary_q names {'정부': 213, '위원': 14, '문재인': 10, '000': 10, '이용': 8, '김정': 5, '박근혜': 5, '이명박': 4}
summary_q session_number {'제2차': 1}


In [24]:
# 이름 오추출 질의 몇 건(질의 원문 일부)
text = {**{q: r["query"] for q, r in frozen.items()}, **{q: r["query"] for q, r in single.items()}}
for _, r in wrong[wrong["field"] == "names"].drop_duplicates("qid").head(6).iterrows():
    i = text[r.qid].find(r.value)
    print(r.set, r.value, "|", text[r.qid][max(0, i - 25): i + 25])

multidoc 정부 | 을 두고 불공정거래행위와의 제재 균형 문제와 정부가 준비 중인 대안은 어떻게 논의되었나?
multidoc 위원 | 2020년 12월 30일 법제사법위원회 법안심사제1소위원회에서 중대재해처벌법안
multidoc 정부 | 법률 개정안(국외 반출 조건부 허용)에 대한 정부 의견은 무엇이었고, 현재 국외 반출 허가
multidoc 위원 | 2017년 11월 15일 교육문화체육관광위원회 예산결산기금심사소위원회의 2018년도 
multidoc 위원 | 2018년 10월 26일 법제사법위원회 국정감사에서 박주민 위원이 군법무관·검
multidoc 정부 | 법안심사제2소위원회에서 항만법 일부개정법률안(정부 제출)의 허가 간주 규정에 대해, 현행 


## 놓친 조건

유형별로 그 유형이 지목에 쓰는 필드가 적중하지 못한 질의를 몇 건 본다. `conf`는 날짜·위원회, `questioner`는 이름, `law`는 날짜·위원회다.

In [25]:
expect = {"conf": ["date", "committee_name"], "questioner": ["names"],
          "law": ["date", "committee_name"]}
md_per = per[per["set"] == "multidoc"]
miss = md_per[md_per.apply(lambda r: r.field in expect[r.type], axis=1) & ~md_per["hit"]]
miss.groupby(["type", "field"]).size()

type  field         
conf  committee_name    7
law   committee_name    4
dtype: int64

In [26]:
for (t, field), g in miss.groupby(["type", "field"]):
    for qid in g["qid"].head(3):
        print(t, field, cond["multidoc"][qid].get(field, []), "|", text[qid][:90])

conf committee_name [] | 2020년 9월 17일 국회 본회의(제382회 제7차) 교육·사회·문화 대정부질문에서 최형두 의원이 추미애 법무부장관 아들 군 휴가 의혹 수사의 8개월 지연과 관
conf committee_name [] | 2017년 2월 10일 국회 본회의(제349회 제7차) 대정부질문에서 오영훈 의원이 제기한 부동산투자이민제도의 법적 근거 문제와 부산 엘시티 지정 특혜 의혹에 대
conf committee_name [] | 2022년 5월 29일 2022년도 제2회 추가경정예산안 등 조정소위원회에서 국채 7.5조 원 상환과 공공자금관리기금의 국채발행 한도(178조 8000억 원) 조
law committee_name [] | 새만금사업 추진 및 지원에 관한 특별법 일부개정법률안 심사에서 2017년 11월 28일 국토법안심사소위원회의 박덕흠 위원과 2019년 7월 17일 국토법안심사소위
law committee_name [] | 서해 5도 지원 특별법 개정안(안상수 의원 대표발의)의 여객선 운항 손실금 지원 조항을 두고 2016년 11월 9일, 2017년 1월 10일, 2017년 8월 2
law committee_name [] | 2003년 국회 본회의 사회·문화에 관한 질문에서 4월 10일 김영환 의원과 6월 11일 민봉기 의원이 법무부장관에게 나라종금 사건 수사에 대해 질의한 내용과 장


## 결함 후보와 수정 효과

위 오추출에서 원래 규칙이 의도와 다르게 동작한 경우를 후보로 둔다. 후보마다 고친 규칙으로 다시 뽑아 G3와 dev에서 바뀌는 질의 수와 정밀도·적중률을 본다. 고칠지는 사용자가 정한다.

- A 이름 사전에 이름이 아닌 값이 있다(`위원`, `소위원장`, `통일부장관`, `정부` 등). 사전 값 중 한글 밖 문자가 있거나, 직함(`ingest.name_titles`)으로 끝나거나, 이름이 아닌 것으로 확인한 값(`정부`, `징세과장`)을 뺀다
- B 두 글자 이름이 일반 낱말과 겹친다(`이용` 등). 두 글자 이름은 뒤에 직함이 올 때만 인정한다
- C `YYYY년도`는 회계연도라 회의 날짜가 아니다. `년도`면 날짜로 뽑지 않는다
- D `제1회 추가경정예산`처럼 회기가 아닌 `제N회`가 뽑힌다. 코퍼스 회기 번호는 180\~402라 세 자리만 회기로 본다

In [27]:
import re

from rag.conditions import DATE, MEETING_NUMBER, Extractor, date_keys
from rag.index import load_cfg

cfg = load_cfg()
meta = json.loads((ROOT / "data/processed/meta_dict.json").read_text(encoding="utf-8"))
titles, suffixes = tuple(cfg.ingest.name_titles), list(cfg.conditions.name_suffixes)
print("정부" in meta["name"], meta["name"].get("정부"), sum(len(n) == 2 for n in meta["name"]))

True 1 129


In [28]:
NOT_NAMES = {"정부", "징세과장"}
clean = {n: c for n, c in meta["name"].items()
         if re.fullmatch("[가-힣]+", n) and not n.endswith(titles) and n not in NOT_NAMES}
print("A로 빠지는 사전 값:", sorted(set(meta["name"]) - set(clean)))
DATE_C = re.compile(r"(\d{4})\s*년(?!도)(?:\s*(\d{1,2})\s*월(?:\s*(\d{1,2})\s*일)?)?")
MEETING_D = re.compile(r"제\s*(\d{3})\s*회(?!의)")


def variant(fixes):
    m = dict(meta, name=clean if "A" in fixes else meta["name"])
    ex = Extractor(m, titles, suffixes)
    if "B" in fixes:
        tail = "|".join(re.escape(t) for t in sorted(titles, key=len, reverse=True))
        long = [n for n in m["name"] if len(n) > 2]
        two = [n for n in m["name"] if len(n) == 2]
        pat = "|".join(re.escape(n) for n in sorted(long, key=len, reverse=True))
        rest = "|".join(re.escape(t) for t in sorted([*titles, *suffixes], key=len, reverse=True))
        ex.name = re.compile(f"(?<![가-힣])(?:({pat})(?=$|[^가-힣]|{rest})|"
                             rf"({'|'.join(map(re.escape, two))})(?=\s*(?:{tail})))")

    def run(text):
        c = ex.extract(text)
        if "B" in fixes:
            got = [a or b for a, b in ex.name.findall(text)]
            c.pop("names", None)
            if got:
                c["names"] = list(dict.fromkeys(got))
        rules = [("date", DATE_C if "C" in fixes else DATE, lambda g: date_keys(*g)[-1]),
                 ("meeting_number", MEETING_D if "D" in fixes else MEETING_NUMBER,
                  lambda g: f"제{g[0]}회")]
        for field, pat, fmt in rules:
            vals = list(dict.fromkeys(fmt(m.groups()) for m in pat.finditer(text)))
            c.pop(field, None)
            if vals:
                c[field] = vals
        return c
    return run

A로 빠지는 사전 값: ['000', '\\김황식', 'ㅌ김현숙', '류지영위원', '소위원장', '위원', '정부', '징세과장', '통일부장관']


In [29]:
def score(run, fields):
    out = []
    for name in ["multidoc", "summary_q"]:
        changed = 0
        for qid, (docs, qtype) in gold[name].items():
            c = run(text[qid])
            changed += c != cond[name][qid]
            for field in fields:
                got = c.get(field, [])
                ok = [v for v in got if any(v in values[d][field] for d in docs)]
                out.append({"set": name, "type": qtype, "field": field,
                            "fired": bool(got), "n": len(got), "n_ok": len(ok), "hit": bool(ok)})
        print(name, "바뀐 질의", changed)
    return table(pd.DataFrame(out).assign(qid=0))


base = variant(set())
assert all(base(text[q]) == cond[n][q] for n in gold for q in gold[n])  # 재현 확인
CANDIDATES = [("A", ["names"]), ("AB", ["names"]), ("C", ["date"]), ("D", ["meeting_number"])]
for fixes, fields in CANDIDATES:
    print("== 후보", fixes)
    display(score(variant(set(fixes)), fields))

== 후보 A
multidoc 바뀐 질의 294
summary_q 바뀐 질의 237


질의    발화율    정밀도    적중률
set       type       field                           
multidoc  conf       names   233  0.296  0.848  0.245
          law        names   152  0.211  0.439  0.079
          questioner names   215  1.000  0.991  1.000
summary_q single     names  3016  0.040  0.152  0.007

== 후보 AB
multidoc 바뀐 질의 298
summary_q 바뀐 질의 259


질의    발화율    정밀도    적중률
set       type       field                           
multidoc  conf       names   233  0.292  0.863  0.245
          law        names   152  0.191  0.474  0.079
          questioner names   215  1.000  0.991  1.000
summary_q single     names  3016  0.032  0.167  0.006

== 후보 C
multidoc 바뀐 질의 54
summary_q 바뀐 질의 1


질의   발화율    정밀도    적중률
set       type       field                          
multidoc  conf       date    233  1.00  0.987  1.000
          law        date    152  1.00  1.000  1.000
          questioner date    215  1.00  0.996  1.000
summary_q single     date   3016  0.01  0.135  0.002

== 후보 D
multidoc 바뀐 질의 17
summary_q 바뀐 질의 0


질의    발화율  정밀도    적중률
set       type       field                                  
multidoc  conf       meeting_number   233  0.245  1.0  0.245
          law        meeting_number   152  0.539  1.0  0.539
          questioner meeting_number   215  0.447  1.0  0.447
summary_q single     meeting_number  3016  0.000  NaN  0.000

## 관찰

- 첫 규칙(G3 측정 1회): 날짜·위원회·회의 종류는 multi-doc 세 유형 모두 발화율 0.91 이상, 정밀도 0.91 이상이다. 국회 대수는 한 번도 뽑히지 않았다
- 이름 정밀도가 낮다(`conf` 0.48, `law` 0.28, `questioner` 0.52). 틀린 이름 348개 중 `위원` 247, `정부` 49로, 사전에 이름이 아닌 값이 들어 있다(S10 사전은 라벨의 이름 칸을 그대로 모았다). dev single-doc에서도 이름이 11.3% 질의에서 뽑히고 정밀도는 0.054다(`정부` 213)
- 날짜 오추출 44개는 대부분 연도만이다(`2018년도 예산`처럼 회계연도). 회기 오추출 17개는 `제1회 추가경정예산`처럼 회기가 아닌 `제N회`다
- 위원회를 놓친 11건은 `국회 본회의`(코퍼스 값 `국회본회의`), `국토법안심사소위원회` 같은 소위원회 이름이다. 본회의는 회의 종류로 뽑혀 있다
- 결함 후보 효과(위 표): A(이름이 아닌 사전 값 제외)는 이름 정밀도를 `conf` 0.85, `questioner` 0.99, `law` 0.44로, single-doc 발화율을 4.0%로 바꾸고 적중률은 그대로다. B(두 글자 이름은 직함 앞에서만)는 A 위에 정밀도를 0.01\~0.03 더 올린다. C(`년도` 제외)는 날짜 정밀도를 0.99 이상으로, D(세 자리 회기만)는 회기 정밀도를 1.0으로 바꾼다
- A 뒤에도 `law` 이름 정밀도는 0.44, single-doc 이름 정밀도는 0.15다. 질의가 화자가 아닌 사람(법안 대표발의자, 대통령 등)을 언급한 경우로, 약한 필터로 쓰는 설계가 전제한 경우다(D-19)
